# API usage Notebook 
- This notebook shows the implementation of each function from the respective libraries.

In [1]:
%load_ext autoreload
%autoreload 2

import dotenv
import os
import pandas as pd
import numpy as np

# Load environment variables (ensure OPENAI_API_KEY is set in your .env)
dotenv.load_dotenv()

# Import the schema agent modules
import research.agentic_data_science.schema_agent.schema_agent_loader as radsasal
import research.agentic_data_science.schema_agent.schema_agent_stats as radsasas
import research.agentic_data_science.schema_agent.schema_agent_hllmcli as radsasah
import research.agentic_data_science.schema_agent.schema_agent_report as radsasar

## 1. Create dummy Dataset

In [2]:
# 1. Create a dummy dataset
np.random.seed(42)
num_rows = 100

dummy_data = pd.DataFrame({
    "employee_id": range(1000, 1000 + num_rows),
    "department": np.random.choice(["Engineering", "Sales", "HR", "Marketing"], num_rows),
    "salary": np.random.normal(85000, 20000, num_rows),
    "satisfaction_score": np.random.uniform(1.0, 5.0, num_rows),
    "hire_date": pd.date_range(start="2018-01-01", periods=num_rows, freq="W").astype(str),
    "notes": ["Good performance"] * 50 + [None] * 50  # 50% nulls
})

# Inject some missing values into salary
dummy_data.loc[10:20, "salary"] = np.nan

# Save to CSV
csv_path = "dummy_employees.csv"
dummy_data.to_csv(csv_path, index=False)
print(f"Created dummy dataset at: {csv_path}")
dummy_data.head()

csv_paths = [csv_path]
tags = ["dummy_employees"]

Created dummy dataset at: dummy_employees.csv


## 2. Load and Infer datatypes from the columns

In [3]:
# 1. Load and prepare DataFrames - now receiving 3 variables
tag_to_df, cat_cols_map, datetime_meta = radsasal.prepare_dataframes(csv_paths, tags)

print("--- Loaded DataFrames ---")
# The index will now show as a DatetimeIndex instead of a RangeIndex
print(tag_to_df["dummy_employees"].info())

# 2. Combine DataFrames while preserving the index
# We do NOT use ignore_index=True here because we want to keep the DatetimeIndex 
# we just created in the loader.
updated_df = pd.concat(list(tag_to_df.values()), axis=0)

print("\n--- Datetime Inference Metadata ---")
# This will now correctly show your temporal column info
print(datetime_meta)

--- Loaded DataFrames ---
<class 'pandas.DataFrame'>
DatetimeIndex: 100 entries, 2018-01-07 00:00:00+00:00 to 2019-12-01 00:00:00+00:00
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   employee_id         100 non-null    int64              
 1   department          100 non-null    str                
 2   salary              89 non-null     float64            
 3   satisfaction_score  100 non-null    float64            
 4   hire_date           100 non-null    datetime64[us, UTC]
 5   notes               50 non-null     str                
dtypes: datetime64[us, UTC](1), float64(2), int64(1), str(2)
memory usage: 5.5 KB
None

--- Datetime Inference Metadata ---
{'hire_date': {'semantic_type': 'temporal', 'granularity': 'date', 'format': 'inferred', 'confidence': 1.0}}


/git_root/research/agentic_data_science/schema_agent/schema_agent_loader.py:75: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce", utc=True)


## 3. Statistical Profiling

In [4]:
# We pass the metadata we just generated into the stats function
stats = radsasas.compute_llm_agent_stats(
    tag_to_df=tag_to_df,
    categorical_cols_map=cat_cols_map,
    metrics=["mean", "std", "min", "max"]
)

# Manually ensure the datetime_columns key is populated for the LLM
stats["datetime_columns"] = datetime_meta

print("\n--- Stats Computation Complete ---")
print(f"Calculated stats for tags: {list(stats['numeric_summary'].keys())}")


=== Temporal Boundaries ===
                                 min_index                 max_index           min_valid_index           max_valid_index
dummy_employees 2018-01-07 00:00:00+00:00 2019-12-01 00:00:00+00:00 2018-01-07 00:00:00+00:00 2018-12-16 00:00:00+00:00
                          employee_id         salary satisfaction_score
2018-01-07 00:00:00+00:00        1000     99769.3316           4.268889
2018-01-14 00:00:00+00:00        1001   88427.365624           3.220803
...                               ...            ...                ...
2019-11-24 00:00:00+00:00        1098  101270.344347           3.313121
2019-12-01 00:00:00+00:00        1099   60382.713671           1.143769
                    num_rows  num_zeros zeros [%]  num_nans nans [%]  num_infs infs [%]  num_valid valid [%]
employee_id              100          0       0.0         0      0.0         0      0.0        100     100.0
salary                   100          0       0.0        11     11.0         0  

## 4. Call LLM for column type inferencing

In [5]:
# 1. Select columns (e.g., let's just send everything)
columns_for_llm = radsasah._select_columns_for_llm(updated_df, scope="all")
print(f"Selected columns for LLM: {columns_for_llm}\n")

# 2. Build the exact prompt string that goes to the LLM
prompt_text = radsasah.build_llm_prompt(stats, columns_to_include=columns_for_llm)
print("--- LLM Prompt Snippet ---")
print(prompt_text[:500] + "\n...\n")

# 3. Call the LLM to generate hypotheses (using gpt-4o as default)
# If you don't have an API key configured, you can mock this response by creating a static dict.
try:
    semantic_insights = radsasah.generate_hypotheses_via_cli(
        stats=stats,
        model="gpt-4o",
        columns_to_include=columns_for_llm
    )
    print("--- LLM Insights Retrieved Successfully ---")
except Exception as e:
    print(f"LLM call failed (Check API key): {e}")
    semantic_insights = {"columns": {}} # Fallback empty dict

Cache hit for apply_llm


Selected columns for LLM: ['employee_id', 'department', 'salary', 'satisfaction_score', 'hire_date', 'notes']

--- LLM Prompt Snippet ---
You are a Senior Data Scientist and Domain Expert.
Analyze the provided dataset statistics and generate a profile for each column.
For each column, provide 2-3 testable hypotheses.
Example: 'Higher discount rates correlate with higher volume but lower margins.'

--- DATASET STATISTICS ---

Detected Datetime Columns:
{
  "hire_date": {
    "semantic_type": "temporal",
    "granularity": "date",
    "format": "inferred",
    "confidence": 1.0
  }
}

Dataset [dummy_employees] Numeric Summary:
     
...

--- LLM Insights Retrieved Successfully ---


## 5. Export to JSON and Markdown

In [6]:
# 1. Build structured column profiles
primary_df = list(tag_to_df.values())[0]
column_profiles = radsasar.build_column_profiles(
    df=primary_df,
    stats=stats,
    insights=semantic_insights
)

# 2. Export to JSON
json_out = "dummy_profile_report.json"
radsasar.merge_and_export_results(
    stats=stats,
    insights=semantic_insights,
    column_profiles=column_profiles,
    output_path=json_out
)

# 3. Export to Markdown
md_out = "dummy_profile_summary.md"
radsasar.export_markdown_from_profiles(
    column_profiles=column_profiles,
    numeric_stats=stats.get("numeric_summary", {}),
    output_path=md_out
)

print(f"\nPipeline complete! Check your directory for:")
print(f"1. {json_out}")
print(f"2. {md_out}")

# Clean up dummy CSV if desired
# os.remove(csv_path)


Pipeline complete! Check your directory for:
1. dummy_profile_report.json
2. dummy_profile_summary.md
